In [1]:
import warnings, os
warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
import matplotlib.pyplot as plt

In [7]:
# Paths - adjust if needed
DATA_PATH = Path("/content/data.csv")  # put your uploaded data here or change to /mnt/data/data.csv
OUT_DIR = Path("figures_trimmed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [8]:
# Load
df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH, "shape:", df.shape)

Loaded: /content/data.csv shape: (569, 33)


In [9]:
# Detect target: common names; fallback to last column
possible = ['target','Outcome','outcome','label','y','risk','at_risk','diagnosis']
target_col = None
for p in possible:
    if p in df.columns:
        target_col = p
        break
if target_col is None:
    target_col = df.columns[-1]
print("Using target column:", target_col)

Using target column: diagnosis


In [10]:
# Map to binary 0/1 if needed
y_raw = df[target_col]
if y_raw.dtype == object or y_raw.dtype.name == 'category':
    # common breast-cancer labels M/B -> 1/0
    if set(y_raw.dropna().unique()) <= set(['M','B']) or set([str(v).upper() for v in y_raw.dropna().unique()]) <= set(['M','B']):
        y = y_raw.map({'M':1,'B':0}).astype(int)
    else:
        # generic mapping to ints
        y = pd.factorize(y_raw)[0]
else:
    y = y_raw.astype(int)

In [11]:
# Use numeric features only for speed
X = df.drop(columns=[target_col]).select_dtypes(include=[np.number]).copy()
X = X.fillna(X.median())

In [12]:
# Stratified split (if binary)
stratify_arg = y if len(set(y)) == 2 else None
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=stratify_arg, random_state=42)

In [13]:
# Pipeline
preproc = ColumnTransformer([('num', StandardScaler(), X.columns.tolist())], remainder='drop', sparse_threshold=0)
clf = HistGradientBoostingClassifier(max_iter=60, random_state=42)  # trimmed
pipe = Pipeline([('preproc', preproc), ('clf', clf)])

In [14]:
# Fit
print("Training (trimmed)...")
pipe.fit(X_train, y_train)

Training (trimmed)...


Pipeline(steps=[('preproc',
                 ColumnTransformer(sparse_threshold=0,
                                   transformers=[('num', StandardScaler(),
                                                  ['id', 'radius_mean',
                                                   'texture_mean',
                                                   'perimeter_mean',
                                                   'area_mean',
                                                   'smoothness_mean',
                                                   'compactness_mean',
                                                   'concavity_mean',
                                                   'concave points_mean',
                                                   'symmetry_mean',
                                                   'fractal_dimension_mean',
                                                   'radius_se', 'texture_se',
                                                   'perimeter_se', 'area_se',
                                                   'smoothness_se',
                                                   'compactness_se',
                                                   'concavity_se',
                                                   'concave points_se',
                                                   'symmetry_se',
                                                   'fractal_dimension_se',
                                                   'radius_worst',
                                                   'texture_worst',
                                                   'perimeter_worst',
                                                   'area_worst',
                                                   'smoothness_worst',
                                                   'compactness_worst',
                                                   'concavity_worst',
                                                   'concave points_worst',
                                                   'symmetry_worst', ...])])),
                ('clf',
                 HistGradientBoostingClassifier(max_iter=60, random_state=42))])

In [15]:
# Evaluation
y_pred = pipe.predict(X_test)
y_proba = pipe.predict_proba(X_test)[:,1] if hasattr(pipe.named_steps['clf'], 'predict_proba') else None
print(classification_report(y_test, y_pred, digits=3))
if y_proba is not None and len(set(y))==2:
    try:
        print("ROC AUC:", round(roc_auc_score(y_test, y_proba), 3))
    except Exception:
        pass

              precision    recall  f1-score   support

           0      0.947     1.000     0.973        72
           1      1.000     0.905     0.950        42

    accuracy                          0.965       114
   macro avg      0.974     0.952     0.961       114
weighted avg      0.967     0.965     0.965       114

ROC AUC: 0.997


In [16]:
# Permutation importance (small repeats)
print("Computing permutation importance (n_repeats=5)...")
perm = permutation_importance(pipe, X_test, y_test, n_repeats=5, random_state=42, n_jobs=1)
feat_names = preproc.get_feature_names_out()
perm_df = pd.DataFrame({'feature': feat_names, 'perm_mean': perm.importances_mean, 'perm_std': perm.importances_std})
perm_df = perm_df.sort_values('perm_mean', ascending=False).reset_index(drop=True)
perm_df.to_csv(OUT_DIR/'permutation_importance.csv', index=False)
print("Saved:", OUT_DIR/'permutation_importance.csv')

Computing permutation importance (n_repeats=5)...
Saved: figures_trimmed/permutation_importance.csv


In [17]:
# Top features for PDP/ICE
top_feats = [f.split('__')[-1] for f in perm_df['feature'].tolist()[:2]]
print("Top features:", top_feats)

Top features: ['perimeter_worst', 'texture_worst']


In [22]:
# PDP + ICE for top feature
if top_feats:
    feat = top_feats[0]
    if feat in X_train.columns:
        fig, ax = plt.subplots(figsize=(7,4))
        PartialDependenceDisplay.from_estimator(pipe, X_train, [feat], kind='both', ax=ax, grid_resolution=20)
        ax.set_title(f"PDP + ICE: {feat}")
        fname = OUT_DIR/f'pdp_ice_{feat}.png'
        fig.savefig(fname, bbox_inches='tight', dpi=150)
        plt.close(fig)
        print("Saved:", fname)
    else:
        print("Top feature not in numeric columns:", feat)

Saved: figures_trimmed/pdp_ice_perimeter_worst.png


In [23]:
# 2D PDP for top pair
if len(top_feats) >= 2 and all(f in X_train.columns for f in top_feats[:2]):
    pair = tuple(top_feats[:2])
    fig2, ax2 = plt.subplots(figsize=(6,5))
    PartialDependenceDisplay.from_estimator(pipe, X_train, [pair], kind='average', ax=ax2, grid_resolution=18)
    ax2.set_title(f"2D PDP: {pair[0]} vs {pair[1]}")
    fname2 = OUT_DIR/f'pdp_2d_{pair[0]}_{pair[1]}.png'
    fig2.savefig(fname2, bbox_inches='tight', dpi=150)
    plt.close(fig2)
    print("Saved:", fname2)
else:
    print("Skipping 2D PDP (not enough numeric top features or not present).")

Saved: figures_trimmed/pdp_2d_perimeter_worst_texture_worst.png


In [24]:
# Clinician table
clinician_rows = []
for f in perm_df['feature'].tolist()[:10]:
    clean = f.split('__')[-1]
    clinician_rows.append({'feature': clean,
                           'interpretation': 'Higher values increase predicted risk.',
                           'recommended_action': 'Consider clinical review if elevated.'})
clinician_df = pd.DataFrame(clinician_rows)
clinician_df.to_csv(OUT_DIR/'clinician_table.csv', index=False)
print("Saved:", OUT_DIR/'clinician_table.csv')

print("All done. Files saved to:", OUT_DIR)

Saved: figures_trimmed/clinician_table.csv
All done. Files saved to: figures_trimmed
